# OmniSpeak (Colab)

Đọc văn bản, nhân bản giọng nói, lưu thư viện giọng — chạy trên Google Colab, dùng model [OmniVoice](https://github.com/k2-fsa/OmniVoice) (`k2-fsa/OmniVoice`, Apache-2.0).

`Runtime → Change runtime type → T4 GPU` trước khi chạy. Chạy các cell theo thứ tự từ trên xuống — mọi cell đều an toàn khi chạy lại.


## Cài đặt

In [ ]:
# 1. GPU check
import shutil
import subprocess

if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
else:
    print("Không có GPU — Runtime → Change runtime type → T4 GPU, rồi chạy lại từ đầu.")


In [ ]:
# 2. Cài đặt
import subprocess
import sys

def run(cmd, what=""):
    print("\n$", cmd if isinstance(cmd, str) else " ".join(cmd))
    if subprocess.run(cmd, shell=isinstance(cmd, str)).returncode != 0:
        raise SystemExit(f"Lỗi: {what or cmd}. Xem log phía trên rồi chạy lại cell này.")

run("apt-get -qq update && apt-get -qq install -y ffmpeg libsndfile1")
run([sys.executable, "-m", "pip", "install", "-q",
     "omnivoice", "fastapi", "uvicorn[standard]", "python-multipart", "soundfile"])
run([sys.executable, "-c",
     "from omnivoice import OmniVoice, VoiceClonePrompt; import fastapi, soundfile; print('OK')"])


In [ ]:
# 3. Giao diện web — tải index.html từ GitHub thay vì nhúng trong notebook
import os
import time
import urllib.request
import urllib.error

FRONTEND_DIR = "/content/omnispeak_frontend"
os.makedirs(FRONTEND_DIR, exist_ok=True)

GITHUB_USER = "trkhanh8312-make"
GITHUB_REPO = "omnispeak"
GITHUB_BRANCH = "main"
GITHUB_PATH = "frontend/index.html"  # đường dẫn file trong repo, vd "frontend/index.html"

# thêm ?_=<timestamp> để né cache CDN của raw.githubusercontent.com (~5 phút),
# đảm bảo luôn lấy đúng bản mới nhất bạn vừa commit
FRONTEND_URL = (
    f"https://raw.githubusercontent.com/{GITHUB_USER}/{GITHUB_REPO}/"
    f"{GITHUB_BRANCH}/{GITHUB_PATH}?_={int(time.time())}"
)

dest = os.path.join(FRONTEND_DIR, "index.html")
try:
    urllib.request.urlretrieve(FRONTEND_URL, dest)
    size = os.path.getsize(dest)
    if size < 500:  # file quá nhỏ -> nhiều khả năng GitHub trả về trang lỗi 404 dạng HTML ngắn
        raise RuntimeError(f"File tải về chỉ {size} byte — có vẻ không phải index.html thật.")
    print(f"Đã tải giao diện web từ GitHub ({size} byte).")
except (urllib.error.URLError, urllib.error.HTTPError, RuntimeError) as e:
    raise RuntimeError(
        "Không tải được giao diện web từ GitHub.\n"
        f"URL đã thử: {FRONTEND_URL}\n"
        "Kiểm tra lại:\n"
        "  1) File đã commit & push lên GitHub chưa?\n"
        "  2) Đường dẫn GITHUB_PATH ở đầu cell này có đúng không?\n"
        "  3) Repo có để public không (raw.githubusercontent.com không đọc được repo private)?\n"
        f"Lỗi gốc: {e}"
    )


In [ ]:
# 4. Mount Google Drive (tuỳ chọn) — lưu bền vững giọng nói + model đã tải
import os

from google.colab import drive

drive.mount("/content/drive")

DATA_DIR = "/content/drive/MyDrive/omnispeak_data"
HF_DIR = "/content/drive/MyDrive/omnispeak_hf_cache"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(HF_DIR, exist_ok=True)
os.environ["OMNISPEAK_DATA_DIR"] = DATA_DIR
os.environ["HF_HOME"] = HF_DIR
os.environ["HF_HUB_DISABLE_SYMLINKS"] = "1"  # Drive (FUSE) không hỗ trợ symlink
print("Giọng nói + model sẽ lưu bền vững trên Drive.")

In [ ]:
# 5. Tải trước model (bỏ qua cũng được — sẽ tự tải khi khởi động backend)
from huggingface_hub import snapshot_download

print("Model tại:", snapshot_download("k2-fsa/OmniVoice"))


In [ ]:
# 6. Backend API — FastAPI gọi thẳng OmniVoice
import os

APP_DIR = "/content/omnispeak_app"
os.makedirs(APP_DIR, exist_ok=True)

BACKEND_PY = 'import io\nimport json\nimport logging\nimport os\nimport hmac\nimport re\nimport threading\nimport time\nimport traceback\nimport uuid\nfrom pathlib import Path\nfrom typing import Optional\n\nimport numpy as np\nimport soundfile as sf\nimport torch\nfrom fastapi import BackgroundTasks, Depends, FastAPI, Form, Header, UploadFile, File, HTTPException\nfrom fastapi.responses import Response\nfrom fastapi.staticfiles import StaticFiles\nfrom omnivoice import OmniVoice, VoiceClonePrompt\n\nDATA_DIR = Path(os.environ.get("OMNISPEAK_DATA_DIR", "/content/omnispeak_data"))\nPROFILES_DIR = DATA_DIR / "profiles"\nINDEX_PATH = DATA_DIR / "profiles.json"\nFRONTEND_DIR = os.environ.get("OMNISPEAK_FRONTEND_DIR", "/content/omnispeak_frontend")\nPROFILES_DIR.mkdir(parents=True, exist_ok=True)\n\n# ---- Log có timestamp/level rõ ràng (in ra stdout, cell 7 đã redirect vào omnispeak_backend.log) ----\nlogging.basicConfig(\n    level=logging.INFO,\n    format="%(asctime)s [%(levelname)s] %(message)s",\n    datefmt="%Y-%m-%d %H:%M:%S",\n)\nlogger = logging.getLogger("omnispeak")\n\n# Giới hạn văn bản: người dùng thường dùng ~1000-2000 từ, chặn cứng ở 3000 từ.\nHARD_WORD_LIMIT = 3000\nCHUNK_MAX_CHARS = 400  # mỗi lần gọi model.generate() xử lý tối đa ~400 ký tự\nSR = 24000\nGAP_SECONDS = 0.25  # khoảng lặng chèn giữa các đoạn khi ghép lại\n\n# Giới hạn file mẫu giọng khi upload\nMAX_UPLOAD_BYTES = 15 * 1024 * 1024  # 15 MB\nMIN_REF_SECONDS = 0.5\nMAX_REF_SECONDS = 120\nPREVIEW_SECONDS = 8       # đoạn mẫu ngắn giữ lại để nghe thử trong thư viện\nMAX_PROFILES = 20         # giới hạn số giọng lưu trong thư viện\nNAME_MAX_LEN = 80         # giới hạn độ dài tên giọng nói\nREF_TEXT_MAX_LEN = 500    # giới hạn độ dài ref_text (transcript của mẫu giọng)\n\n# Dọn job cũ khỏi bộ nhớ\nJOB_RESULT_TTL = 30 * 60     # job đã xong nhưng không ai lấy audio -> xoá sau 30 phút\nJOB_STALE_TTL = 2 * 60 * 60  # job kẹt bất thường quá 2 tiếng -> xoá luôn (an toàn)\n\n# profile_id / job_id đều là uuid4().hex[:12] -> chỉ gồm hex 12 ký tự.\n# Bắt buộc khớp định dạng này trước khi ghép vào đường dẫn file, tránh path traversal\n# (vd ai đó gửi profile_id="../../etc/passwd" qua URL).\nID_RE = re.compile(r"^[0-9a-f]{12}$")\n\n\ndef _validate_id(id_: str):\n    if not ID_RE.fullmatch(id_ or ""):\n        raise HTTPException(400, "ID không hợp lệ")\n\n\n# ---- Khoa bao ve API bang secret key (dat trong Colab Secrets: OMNISPEAK_SECRET) ----\n# Cach dat: icon khoa "Secrets" o sidebar trai cua Colab -> them ten "OMNISPEAK_SECRET",\n# gia tri tuy ban chon. Neu chua dat, API tam thoi KHONG bi khoa (de tranh tu khoa ban ra ngoai).\nAPP_SECRET = os.environ.get("OMNISPEAK_SECRET", "").strip()\nif not APP_SECRET:\n    logger.warning("OMNISPEAK_SECRET chua duoc dat - API dang chay KHONG co mat khau bao ve.")\n\n\ndef require_key(x_omnispeak_key: str = Header(default="")):\n    if not APP_SECRET:\n        return\n    if not hmac.compare_digest(x_omnispeak_key, APP_SECRET):\n        raise HTTPException(401, "Sai mat khau")\n\n\napp = FastAPI()\n\ndevice = "cuda:0" if torch.cuda.is_available() else "cpu"\ndtype = torch.float16 if torch.cuda.is_available() else torch.float32\nlogger.info(f"Đang tải model OmniVoice lên {device}...")\nMODEL = OmniVoice.from_pretrained("k2-fsa/OmniVoice", device_map=device, dtype=dtype, load_asr=True)\nlogger.info("Model đã sẵn sàng.")\n\nGEN_LOCK = threading.Lock()  # chỉ chạy 1 job trên GPU tại một thời điểm\nJOBS: dict = {}  # job_id -> {status, total, done, audio_bytes, gen_time, error, created}\n\n\ndef _cleanup_jobs():\n    """Xoá job đã xong lâu mà không ai lấy kết quả, và job kẹt bất thường quá lâu."""\n    now = time.time()\n    stale = []\n    for jid, job in JOBS.items():\n        age = now - job.get("created", now)\n        if job["status"] in ("done", "error") and age > JOB_RESULT_TTL:\n            stale.append(jid)\n        elif age > JOB_STALE_TTL:\n            stale.append(jid)\n    for jid in stale:\n        JOBS.pop(jid, None)\n    if stale:\n        logger.info(f"Đã dọn {len(stale)} job cũ khỏi bộ nhớ.")\n\n\ndef _has_active_job():\n    return any(j["status"] in ("pending", "running") for j in JOBS.values())\n\n\ndef _load_index():\n    return json.loads(INDEX_PATH.read_text()) if INDEX_PATH.exists() else []\n\n\ndef _save_index(items):\n    INDEX_PATH.write_text(json.dumps(items, ensure_ascii=False, indent=2))\n\n\ndef _split_into_chunks(text: str, max_chars: int = CHUNK_MAX_CHARS):\n    """Tách văn bản thành các đoạn nhỏ theo câu, mỗi đoạn tối đa ~max_chars ký tự."""\n    sentences = re.split(r"(?<=[.!?…])\\s+", text.strip())\n    chunks, buf = [], ""\n    for s in sentences:\n        s = s.strip()\n        if not s:\n            continue\n        if len(s) > max_chars:\n            # câu quá dài — cắt theo khoảng trắng\n            words = s.split(" ")\n            piece = ""\n            for w in words:\n                if len(piece) + len(w) + 1 > max_chars:\n                    if piece:\n                        chunks.append(piece.strip())\n                    piece = w\n                else:\n                    piece = (piece + " " + w).strip()\n            if piece:\n                s = piece\n            else:\n                continue\n        if len(buf) + len(s) + 1 <= max_chars:\n            buf = (buf + " " + s).strip()\n        else:\n            if buf:\n                chunks.append(buf)\n            buf = s\n    if buf:\n        chunks.append(buf)\n    return chunks or [text.strip()]\n\n\ndef _run_job(job_id: str, text: str, profile_id: Optional[str]):\n    job = JOBS[job_id]\n    job["status"] = "running"\n    logger.info(f"[{job_id}] Bắt đầu tạo giọng nói — {len(text)} ký tự, profile={profile_id or \'mặc định\'}")\n    try:\n        kwargs = {}\n        if profile_id:\n            p = PROFILES_DIR / f"{profile_id}.pt"\n            if not p.exists():\n                raise RuntimeError(f"Profile {profile_id} not found")\n            kwargs["voice_clone_prompt"] = VoiceClonePrompt.load(str(p))\n\n        chunks = _split_into_chunks(text)\n        job["total"] = len(chunks)\n\n        t0 = time.time()\n        pieces = []\n        gap = np.zeros(int(GAP_SECONDS * SR), dtype=np.float32)\n\n        with GEN_LOCK:\n            for i, chunk in enumerate(chunks):\n                audio = MODEL.generate(text=chunk, **kwargs)\n                wav = audio[0]\n                if isinstance(wav, torch.Tensor):\n                    wav = wav.detach().cpu().float().numpy()\n                wav = np.asarray(wav, dtype=np.float32)\n                if wav.ndim == 2:\n                    wav = wav.T\n                    if wav.shape[1] == 1:\n                        wav = wav[:, 0]\n                pieces.append(wav)\n                if i < len(chunks) - 1:\n                    pieces.append(gap)\n                job["done"] = i + 1\n\n        if torch.cuda.is_available():\n            torch.cuda.empty_cache()\n\n        full = np.concatenate(pieces) if len(pieces) > 1 else pieces[0]\n        buf = io.BytesIO()\n        sf.write(buf, full, SR, format="WAV", subtype="PCM_16")\n        job["audio_bytes"] = buf.getvalue()\n        job["gen_time"] = time.time() - t0\n        job["status"] = "done"\n        logger.info(f"[{job_id}] Xong trong {job[\'gen_time\']:.2f}s ({len(chunks)} đoạn).")\n    except Exception as e:\n        job["status"] = "error"\n        job["error"] = str(e)\n        logger.error(f"[{job_id}] Lỗi khi tạo giọng nói: {e}\\n{traceback.format_exc()}")\n\n\n@app.get("/health")\ndef health():\n    return {"status": "ok", "device": "cuda" if torch.cuda.is_available() else "cpu"}\n\n\n@app.post("/generate", dependencies=[Depends(require_key)])\nasync def generate(background_tasks: BackgroundTasks, text: str = Form(...), profile_id: Optional[str] = Form(None)):\n    _cleanup_jobs()\n    text = text.strip()\n    if not text:\n        raise HTTPException(400, "Văn bản trống")\n    word_count = len(text.split())\n    if word_count > HARD_WORD_LIMIT:\n        raise HTTPException(413, f"Văn bản vượt quá {HARD_WORD_LIMIT} từ (hiện tại: {word_count} từ)")\n    if profile_id:\n        _validate_id(profile_id)\n        p = PROFILES_DIR / f"{profile_id}.pt"\n        if not p.exists():\n            raise HTTPException(404, f"Profile {profile_id} not found")\n    if _has_active_job():\n        raise HTTPException(429, "Đang có một yêu cầu tạo giọng khác được xử lý, vui lòng đợi rồi thử lại.")\n\n    chunks = _split_into_chunks(text)\n    job_id = uuid.uuid4().hex[:12]\n    JOBS[job_id] = {"status": "pending", "total": len(chunks), "done": 0,\n                     "audio_bytes": None, "gen_time": None, "error": None,\n                     "created": time.time()}\n    logger.info(f"[{job_id}] Job mới — {word_count} từ, {len(chunks)} đoạn.")\n    background_tasks.add_task(_run_job, job_id, text, profile_id)\n    return {"job_id": job_id, "total_chunks": len(chunks)}\n\n\n@app.get("/generate/{job_id}/status", dependencies=[Depends(require_key)])\ndef generate_status(job_id: str):\n    job = JOBS.get(job_id)\n    if not job:\n        raise HTTPException(404, "Job not found")\n    return {"status": job["status"], "total": job["total"], "done": job["done"], "error": job["error"]}\n\n\n@app.get("/generate/{job_id}/audio", dependencies=[Depends(require_key)])\ndef generate_audio(job_id: str):\n    job = JOBS.get(job_id)\n    if not job:\n        raise HTTPException(404, "Job not found")\n    if job["status"] == "error":\n        raise HTTPException(500, job["error"] or "Lỗi tạo giọng nói")\n    if job["status"] != "done":\n        raise HTTPException(409, "Job chưa xong")\n    data = job["audio_bytes"]\n    gen_time = job["gen_time"]\n    JOBS.pop(job_id, None)  # dọn bộ nhớ sau khi trả kết quả\n    return Response(content=data, media_type="audio/wav",\n                     headers={"X-Gen-Time": f"{gen_time:.2f}"})\n\n\n@app.get("/profiles", dependencies=[Depends(require_key)])\ndef list_profiles():\n    return _load_index()\n\n\n@app.get("/profiles/{profile_id}/preview", dependencies=[Depends(require_key)])\ndef profile_preview(profile_id: str):\n    _validate_id(profile_id)\n    path = PROFILES_DIR / f"{profile_id}_preview.wav"\n    if not path.exists():\n        raise HTTPException(404, "Giọng này chưa có bản nghe thử (được tạo trước khi tính năng này ra mắt).")\n    return Response(content=path.read_bytes(), media_type="audio/wav")\n\n\n@app.post("/profiles", dependencies=[Depends(require_key)])\nasync def create_profile(name: str = Form(...), kind: str = Form("clone"),\n                          ref_audio: UploadFile = File(...), ref_text: Optional[str] = Form(None)):\n    name = name.strip()\n    if not name:\n        raise HTTPException(400, "Tên giọng nói không được để trống")\n    if len(name) > NAME_MAX_LEN:\n        raise HTTPException(400, f"Tên giọng nói vượt quá {NAME_MAX_LEN} ký tự")\n    if ref_text and len(ref_text) > REF_TEXT_MAX_LEN:\n        raise HTTPException(400, f"ref_text vượt quá {REF_TEXT_MAX_LEN} ký tự")\n\n    items = _load_index()\n    existing = next((p for p in items if p["name"] == name), None)\n    if existing:\n        return existing\n    if len(items) >= MAX_PROFILES:\n        raise HTTPException(429, f"Thư viện đã đạt giới hạn {MAX_PROFILES} giọng nói — hãy xoá bớt giọng cũ trước khi thêm mới.")\n\n    raw = await ref_audio.read()\n    if not raw:\n        raise HTTPException(400, "File mẫu giọng trống")\n    if len(raw) > MAX_UPLOAD_BYTES:\n        raise HTTPException(413, f"File mẫu giọng vượt quá {MAX_UPLOAD_BYTES // (1024*1024)}MB")\n\n    profile_id = uuid.uuid4().hex[:12]\n    tmp = DATA_DIR / f"_upload_{profile_id}.wav"\n    tmp.write_bytes(raw)\n\n    try:\n        # Kiểm tra file có phải audio hợp lệ + thời lượng hợp lý trước khi đưa vào model\n        try:\n            info = sf.info(str(tmp))\n        except Exception:\n            raise HTTPException(400, "File mẫu không phải audio hợp lệ (thử .wav, .mp3, .m4a...)")\n        if info.duration < MIN_REF_SECONDS:\n            raise HTTPException(400, f"Mẫu giọng quá ngắn ({info.duration:.1f}s) — cần ít nhất {MIN_REF_SECONDS}s")\n        if info.duration > MAX_REF_SECONDS:\n            raise HTTPException(400, f"Mẫu giọng quá dài ({info.duration:.0f}s) — tối đa {MAX_REF_SECONDS}s")\n\n        # Lưu lại một đoạn mẫu ngắn để nghe thử trong thư viện (không lưu toàn bộ file gốc)\n        try:\n            data, sr = sf.read(str(tmp))\n            preview = data[: int(PREVIEW_SECONDS * sr)]\n            sf.write(str(PROFILES_DIR / f"{profile_id}_preview.wav"), preview, sr, subtype="PCM_16")\n        except Exception as e:\n            logger.error(f"Không tạo được bản xem trước cho \'{name}\': {e}")\n\n        try:\n            prompt = MODEL.create_voice_clone_prompt(ref_audio=str(tmp), ref_text=ref_text or None)\n            prompt.save(str(PROFILES_DIR / f"{profile_id}.pt"))\n        except Exception as e:\n            logger.error(f"Lỗi tạo voice clone prompt cho \'{name}\': {e}\\n{traceback.format_exc()}")\n            (PROFILES_DIR / f"{profile_id}_preview.wav").unlink(missing_ok=True)\n            raise HTTPException(500, f"Không tạo được giọng nói từ mẫu này: {e}")\n    finally:\n        tmp.unlink(missing_ok=True)\n\n    entry = {"id": profile_id, "name": name, "kind": kind}\n    items.append(entry)\n    _save_index(items)\n    logger.info(f"Đã tạo giọng mới: {name} ({profile_id})")\n    return entry\n\n\n@app.delete("/profiles/{profile_id}", dependencies=[Depends(require_key)])\ndef delete_profile(profile_id: str):\n    _validate_id(profile_id)\n    items = _load_index()\n    remaining = [p for p in items if p["id"] != profile_id]\n    if len(remaining) == len(items):\n        raise HTTPException(404, "Not found")\n    _save_index(remaining)\n    (PROFILES_DIR / f"{profile_id}.pt").unlink(missing_ok=True)\n    (PROFILES_DIR / f"{profile_id}_preview.wav").unlink(missing_ok=True)\n    logger.info(f"Đã xoá giọng: {profile_id}")\n    return {"deleted": profile_id}\n\n\napp.mount("/", StaticFiles(directory=FRONTEND_DIR, html=True), name="frontend")\n'

with open(os.path.join(APP_DIR, "backend.py"), "w", encoding="utf-8") as f:
    f.write(BACKEND_PY)
print("Đã ghi backend.py.")


In [ ]:
# 7. Khởi động backend
FORCE_RESTART = True  # luôn nạp lại code mới nhất từ cell 6

import json
import os
import subprocess
import sys
import time
import urllib.request

APP_DIR = "/content/omnispeak_app"
PORT = 3900
LOG_PATH = "/content/omnispeak_backend.log"
HEALTH_URL = f"http://127.0.0.1:{PORT}/health"

def health():
    try:
        with urllib.request.urlopen(HEALTH_URL, timeout=5) as r:
            return json.load(r)
    except Exception:
        return None

info = None if FORCE_RESTART else health()
if info:
    print("Backend đang chạy —", info)
else:
    subprocess.run(f"kill -9 $(lsof -t -i:{PORT}) 2>/dev/null || true", shell=True)
    time.sleep(1)

    env = os.environ.copy()
    env["OMNISPEAK_DATA_DIR"] = os.environ.get("OMNISPEAK_DATA_DIR", "/content/omnispeak_data")
    env["OMNISPEAK_FRONTEND_DIR"] = "/content/omnispeak_frontend"
    env["PYTHONUNBUFFERED"] = "1"

    try:
        from google.colab import userdata
        env["OMNISPEAK_SECRET"] = userdata.get("OMNISPEAK_SECRET")
        print("Da doc OMNISPEAK_SECRET tu Colab Secrets.")
    except Exception:
        env["OMNISPEAK_SECRET"] = ""
        print("Chua dat Colab Secret 'OMNISPEAK_SECRET' - backend se chay KHONG co mat khau bao ve.")

    log = open(LOG_PATH, "ab")
    proc = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "backend:app", "--app-dir", APP_DIR,
         "--host", "127.0.0.1", "--port", str(PORT)],
        env=env, stdout=log, stderr=subprocess.STDOUT,
    )
    print(f"Đang khởi động (PID {proc.pid})...")

    deadline = time.time() + 300
    while time.time() < deadline:
        if proc.poll() is not None:
            break
        info = health()
        if info:
            break
        print(".", end="", flush=True)
        time.sleep(3)
    print()

    if info:
        print("Backend đã sẵn sàng —", info)
    else:
        try:
            tail = "".join(open(LOG_PATH, errors="replace").readlines()[-40:])
        except OSError:
            tail = "(không có log)"
        raise SystemExit(f"Backend không lên được sau 5 phút.\n--- log ---\n{tail}")


In [ ]:
# 8. Mở giao diện web
from google.colab import output

output.serve_kernel_port_as_window(3900)


### Tổng kết

Danh sách giọng nói đã lưu.

In [ ]:
# Tổng kết
import requests

try:
    profiles = requests.get("http://127.0.0.1:3900/profiles", timeout=15).json()
    print(f"Giọng đã lưu ({len(profiles)}):")
    for p in profiles:
        print(" ", p["id"], p.get("name"))
except Exception as e:
    print("Không lấy được danh sách:", e)


## Xử lý sự cố

- **`device: cpu` hoặc generate chậm** — bật GPU: Runtime → Change runtime type → T4 GPU.
- **Sửa `backend.py` (cell 6) xong mà lỗi vẫn y nguyên** — cell 7 có `FORCE_RESTART = True`, chạy lại cell 7 là tự nạp code mới, không cần tự kill process.
- **Muốn giữ giọng nói + model qua các phiên sau** — chạy cell 4 (mount Drive) trước cell 5.
- **Tab UI trắng hoặc lỗi** — chạy lại cell 7 rồi cell 8. Cho phép pop-up cho `colab.research.google.com`; hoặc đổi cell 8 sang `output.serve_kernel_port_as_iframe(3900)` để nhúng UI ngay trong notebook.
- **Xem log backend** — `/content/omnispeak_backend.log`.
